# LangSmith Client Anonymizer — PII Redaction in Tool Calls

This notebook demonstrates the **Client anonymizer** approach to PII masking.
The anonymizer sits on the LangSmith `Client` and scrubs PII from **all** traced data —
user inputs, tool call args, tool results, and AI responses — regardless of graph structure.

We'll:
1. Run a LangChain agent that calls a `lookup_customer` tool returning PII
2. Pull the trace from LangSmith to verify everything is redacted

## Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

# Ensure tracing is on
os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_PROJECT", "pii-masking-agent")

'pii-masking-agent'

## Create the Anonymizer + LangSmith Client

One `Client(anonymizer=...)` covers all traced data. No per-field middleware flags needed.

In [2]:
from langsmith import Client
from langsmith.anonymizer import create_anonymizer

anonymizer = create_anonymizer([
    { "pattern": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "replace": "[EMAIL_REDACTED]" },
    { "pattern": r"\b(?:\d{1,3}\.){3}\d{1,3}\b", "replace": "[IP_REDACTED]" },
    { "pattern": r"(\+\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}", "replace": "[PHONE_REDACTED]" },
    { "pattern": r"\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b", "replace": "[CC_REDACTED]" },
    { "pattern": r"\b\d{3}-\d{2}-\d{4}\b", "replace": "[SSN_REDACTED]" },
    { "pattern": r"\b\d{1,2}/\d{1,2}/\d{4}\b|\b\d{4}-\d{1,2}-\d{1,2}\b", "replace": "[DATE_REDACTED]" },
])

langsmith_client = Client(anonymizer=anonymizer)

## Define Tools

The `lookup_customer` tool returns a record full of PII — email, phone, SSN, credit card, IP.

In [3]:
from langchain_core.tools import tool

CUSTOMER_DB = {
    "john smith": {
        "name": "John Smith",
        "email": "john.smith@acmecorp.com",
        "phone": "(555) 867-5309",
        "ssn": "123-45-6789",
        "credit_card": "4532-1488-0343-6728",
        "ip_address": "192.168.1.42",
        "address": "123 Main St, Springfield, IL 62701",
    },
    "jane doe": {
        "name": "Jane Doe",
        "email": "jane.doe@globex.net",
        "phone": "(555) 234-5678",
        "ssn": "987-65-4321",
        "credit_card": "5105-1051-0510-5100",
        "ip_address": "10.0.0.7",
        "address": "456 Oak Ave, Shelbyville, IL 62565",
    },
}

@tool
def lookup_customer(name: str) -> str:
    """Look up customer information by name. Returns their full profile."""
    key = name.strip().lower()
    customer = CUSTOMER_DB.get(key)
    if not customer:
        return f"No customer found with name '{name}'."
    return (
        f"Customer Record:\n"
        f"  Name: {customer['name']}\n"
        f"  Email: {customer['email']}\n"
        f"  Phone: {customer['phone']}\n"
        f"  SSN: {customer['ssn']}\n"
        f"  Credit Card: {customer['credit_card']}\n"
        f"  IP Address: {customer['ip_address']}\n"
        f"  Address: {customer['address']}"
    )

## Create and Run the Agent

We wrap the agent invocation in `ls.tracing_context(client=langsmith_client)` so the
anonymizer scrubs all traced data.

In [4]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import langsmith as ls

agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[lookup_customer],
)

# Run with the anonymizer active on all traces
with ls.tracing_context(client=langsmith_client):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Look up the customer record for John Smith"}]},
    )

# Print the agent's final response
print(result["messages"][-1].content)

Here is the customer record for John Smith:

- **Name:** John Smith
- **Email:** john.smith@acmecorp.com
- **Phone:** (555) 867-5309
- **SSN:** 123-45-6789
- **Credit Card:** 4532-1488-0343-6728
- **IP Address:** 192.168.1.42
- **Address:** 123 Main St, Springfield, IL 62701


## Pull the Trace from LangSmith

Now let's fetch the most recent trace and inspect the traced inputs/outputs
to confirm PII was redacted before it ever reached LangSmith.

In [5]:
import time

# Give LangSmith a moment to ingest the trace
time.sleep(5)

# Use a regular client (no anonymizer) to read back what LangSmith actually stored
reader_client = Client()

project_name = os.environ.get("LANGSMITH_PROJECT", "pii-masking-agent")

# Get the most recent trace
runs = list(reader_client.list_runs(
    project_name=project_name,
    is_root=True,
    limit=1,
))

if not runs:
    print("No traces found. Check your LANGSMITH_API_KEY and project name.")
else:
    root_run = runs[0]
    print(f"Trace ID: {root_run.id}")
    print(f"Trace URL: {root_run.url}\n")

Trace ID: 019c7ce1-1361-7720-b56d-f4c12b71b1a1
Trace URL: https://smith.langchain.com/o/dbfd3524-0620-4331-9e67-af8255f2cc8b/projects/p/63359a66-ae1c-4e8a-a06b-d6b5e6a16549/r/019c7ce1-1361-7720-b56d-f4c12b71b1a1?trace_id=019c7ce1-1361-7720-b56d-f4c12b71b1a1&start_time=2026-02-20T21:07:31.297154



In [6]:
import json

# Fetch all spans for this trace
spans = list(reader_client.list_runs(
    trace_id=root_run.trace_id,
    project_name=project_name,
))

# Raw PII that should NOT appear in LangSmith — the anonymizer should have replaced these
raw_pii = {
    "email":       "john.smith@acmecorp.com",
    "phone":       "(555) 867-5309",
    "ssn":         "123-45-6789",
    "credit_card": "4532-1488-0343-6728",
    "ip_address":  "192.168.1.42",
}

print(f"Checking {len(spans)} spans for raw PII...\n")

leaks_found = []

for span in spans:
    traced = json.dumps({"inputs": span.inputs, "outputs": span.outputs})
    for pii_type, pii_value in raw_pii.items():
        if pii_value in traced:
            leaks_found.append((span.name, pii_type, pii_value))

if leaks_found:
    print("⚠️  PII LEAK DETECTED:")
    for span_name, pii_type, pii_value in leaks_found:
        print(f"   Span '{span_name}' contains raw {pii_type}: {pii_value}")
else:
    print("✅ ALL CLEAN — no raw PII found in any traced span.")
    print("\nWhat LangSmith actually stored for the tool result:")
    tool_span = next((s for s in spans if s.run_type == "tool"), None)
    if tool_span:
        print(json.dumps(tool_span.outputs, indent=2))

Checking 7 spans for raw PII...

✅ ALL CLEAN — no raw PII found in any traced span.

What LangSmith actually stored for the tool result:
{
  "output": {
    "additional_kwargs": {},
    "content": "Customer Record:\n  Name: John Smith\n  Email: [EMAIL_REDACTED]\n  Phone: [PHONE_REDACTED]\n  SSN: [SSN_REDACTED]\n  Credit Card: [CC_REDACTED]\n  IP Address: [IP_REDACTED]\n  Address: 123 Main St, Springfield, IL 62701",
    "name": "lookup_customer",
    "response_metadata": {},
    "status": "success",
    "tool_call_id": "call_lb33cX4uzjLOF1klFlyLrFm8",
    "type": "tool"
  }
}
